# Figure S7 - true-genealogy homoplasy and variant confusability

Across-run version of the single-simulation Figure S7, built from the true simulation genealogies (`run-out.branches`) rather than the reconstructed trees. Panel **A** is the origin-count ECDF per site class; panel **B** is the shared-substitution minus null co-assignment per method. Each simulation is a faint curve; the representative run is bold. The spread of the faint curves is the uncertainty (no analytic error bars: origin pairs are not independent).


## Imports and parameters

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

%matplotlib inline

HERE = Path.cwd()
sys.path.append(str(HERE.parent / 'scripts'))

from plot_truetree_homoplasy import build_pooled_figure
from plot_mutation_homoplasy import REPRESENTATIVE_RUN
from antigentools.supplement_style import apply_supplement_style

apply_supplement_style()

BATCH = '2026-07-04-reviewer-runs'
AGG_DIR = HERE.parent / 'results' / 'aggregated' / BATCH
CAND_PATH = HERE.parent / 'data' / BATCH / 'antigen-outputs' / 'candidate_runs.csv'
FIG_DIR = HERE.parent.parent / 'antigen-tex' / 'reviews' / 'round1' / 'figures'
FIG_PREFIX = 'figureS7_truetree_homoplasy'

## Load the sweep output and restrict to the flu-like candidate runs

The sweep covers every run with a data directory; filtering to the candidates happens here, using the same `cand_keys` idiom as figures S3/S4/S6.

In [ ]:
recurrence = pd.read_csv(AGG_DIR / 'truetree_recurrence_by_run.csv')
origin_counts = pd.read_csv(AGG_DIR / 'truetree_origin_counts_by_run.csv')
confusability = pd.read_csv(AGG_DIR / 'truetree_confusability_by_run.csv')

cand = pd.read_csv(CAND_PATH)


def config_from_path(p):
    m = re.search(r'simulations/([^/]+)/run_', p)
    assert m is not None, f'cannot parse config from candidate path: {p!r}'
    return m.group(1)


cand['config'] = cand['path'].map(config_from_path)
cand_keys = set(zip(cand['config'], cand['run'].astype(int)))


def keep_candidates(df, label):
    keys = list(zip(df['config'], df['run'].astype(int)))
    kept = df[[k in cand_keys for k in keys]].copy()
    assert not kept.empty, f'no candidate runs survived the join for {label}'
    print(f'{label}: {len(kept)} rows from {kept.groupby(["config", "run"]).ngroups} candidate runs')
    return kept


recurrence = keep_candidates(recurrence, 'per-run recurrence')
origin_counts = keep_candidates(origin_counts, 'origin-count ECDF')
confusability = keep_candidates(confusability, 'confusability')

# Runs missing per-run variant labels contribute recurrence and ECDF rows but no
# confusability rows; report how many so the panel-B pool size is explicit.
notes = recurrence['notes'].fillna('')
print(f"no tips_with_variants.tsv (panel B excluded): {notes.str.contains('no tips_with_variants').sum()} of {len(recurrence)}")

## Figure S7

In [ ]:
with plt.rc_context({}):
    fig = build_pooled_figure(origin_counts, confusability, REPRESENTATIVE_RUN)
plt.show()

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
for suffix in ('pdf', 'png'):
    path = FIG_DIR / f'{FIG_PREFIX}.{suffix}'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Wrote {path}')

**Figure S7.** True-genealogy homoplasy and variant confusability across all flu-like candidate `antigen-prime` simulations. Each faint curve is one simulation; the representative run is bold.

**(A)** Cumulative fraction of amino-acid substitutions of each site class arising on at most X separate branches of the true genealogy. Recurrence is common and counted exactly, without the inference inflation of a reconstructed tree.

**(B)** For every pair of independent origins of one substitution, the change in the probability that the two origins' descendants receive the same variant label, relative to a matched null of origin pairs from different substitutions at the same background distance. Zero means sharing a substitution adds no co-assignment beyond background similarity; the two distance-based methods sit near zero throughout, and the phylogenetic method adds a modest amount only for near-identical backgrounds.

## Numbers quoted in the response letter

Pooled across candidate runs, weighting by substitution counts rather than averaging per-run rates (runs differ in size).

In [ ]:
def pooled(site):
    return recurrence[f'{site}_n_recurrent'].sum() / recurrence[f'{site}_n'].sum()


for site in ('epitope', 'non_epitope'):
    print(f'{site:12s} recurrence {pooled(site):6.1%}   '
          f'max origins (any run) {int(recurrence[f"{site}_max_origins"].max())}')